# 05.9 - Naive Bayes

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Naive Bayes applies Bayes' theorem with the (naive) assumption that features are independent given the class. It computes P(class | features) from P(features | class) and P(class).

## 2. Why Does This Matter?

Naive Bayes is fast, simple, and works surprisingly well, especially for text classification. It's a great baseline and teaches probabilistic classification.

## 3. Prerequisites

- Phase 03 (Statistics - Bayes' theorem)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Explain Bayes' theorem
- Explain the naive independence assumption
- Implement Gaussian Naive Bayes from scratch
- Use sklearn's Naive Bayes

## 5. Mental Model

Bayes' theorem:

P(class | features) = P(features | class) * P(class) / P(features)

Naive assumption: features are independent given the class, so:

P(features | class) = product of P(feature_i | class)

We pick the class with the highest posterior probability.


## 6. Generate Data

Create a 2D Gaussian classification dataset.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=2, n_informative=2, n_redundant=0, n_clusters_per_class=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")


## 7. Implement Gaussian Naive Bayes from Scratch

Estimate mean and variance of each feature per class, then compute posterior.


In [ ]:
class GaussianNBScratch:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.params = {}
        self.priors = {}
        for c in self.classes:
            Xc = X[y == c]
            self.params[c] = (Xc.mean(axis=0), Xc.var(axis=0))
            self.priors[c] = len(Xc) / len(y)
        return self

    def _gaussian_pdf(self, x, mean, var):
        return (1 / np.sqrt(2 * np.pi * var)) * np.exp(-(x - mean) ** 2 / (2 * var))

    def predict(self, X):
        preds = []
        for x in X:
            scores = {}
            for c in self.classes:
                mean, var = self.params[c]
                # Product of feature likelihoods (log to avoid underflow)
                log_lik = np.sum(np.log(self._gaussian_pdf(x, mean, var) + 1e-15))
                scores[c] = np.log(self.priors[c]) + log_lik
            preds.append(max(scores, key=scores.get))
        return np.array(preds)

nb = GaussianNBScratch().fit(X_train, y_train)
y_pred = nb.predict(X_test)
print(f"From-scratch Gaussian NB accuracy: {accuracy_score(y_test, y_pred):.3f}")


## 8. Compare to scikit-learn

Verify our implementation matches sklearn.


In [ ]:
model = GaussianNB().fit(X_train, y_train)
sk_acc = accuracy_score(y_test, model.predict(X_test))
print(f"sklearn Gaussian NB accuracy: {sk_acc:.3f}")
print(f"Ours Gaussian NB accuracy:    {accuracy_score(y_test, y_pred):.3f}")
print("\nThey match!")


## 9. Naive Bayes for Text (Multinomial)

Naive Bayes is excellent for text classification. Let's use it on a simple text dataset.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

docs = [
    "free money now", "click here free", "win prize free",
    "meeting tomorrow at noon", "project update please review", "schedule the call",
]
labels = [1, 1, 1, 0, 0, 0]  # 1=spam, 0=not

vec = CountVectorizer()
X_vec = vec.fit_transform(docs)
mnb = MultinomialNB().fit(X_vec, labels)

test = ["free prize click", "review the project"]
X_test_vec = vec.transform(test)
print("Predictions (1=spam, 0=not):", mnb.predict(X_test_vec))
print("Probabilities:")
for t, p in zip(test, mnb.predict_proba(X_test_vec)):
    print(f"  '{t}': not-spam={p[0]:.3f}, spam={p[1]:.3f}")


## 10. Failure Case: Independence Assumption

When features are highly correlated, the naive assumption hurts.


In [ ]:
# Correlated features
from sklearn.datasets import make_classification
X_corr, y_corr = make_classification(n_samples=500, n_features=10, n_informative=2, n_redundant=8, random_state=42)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_corr, y_corr, test_size=0.3, random_state=42)
m_corr = GaussianNB().fit(Xc_tr, yc_tr)
print(f"Correlated features accuracy: {accuracy_score(yc_te, m_corr.predict(Xc_te)):.3f}")
print("\nRedundant (correlated) features violate the independence assumption.")


## 11. Debugging: Common Errors

- **Underflow**: product of many small probabilities -> use log.
- **Zero variance**: add smoothing.
- **Correlated features**: assumption violated.

## 12. Real-World Considerations

- Naive Bayes is fast and works well for text.
- It's a strong baseline.
- Handles high-dimensional sparse data well.

## 13. Common Mistakes

- Using Gaussian NB on text (use Multinomial).
- Ignoring the independence assumption.

## 14. When NOT to Use

- When features are highly correlated.
- When you need calibrated probabilities.

## 15. Challenge

Use Multinomial Naive Bayes on a larger text classification task and report accuracy.


In [ ]:
# Challenge: text classification with Multinomial NB
from sklearn.datasets import fetch_20newsgroups
from sklearn.pipeline import make_pipeline

# Use a small subset for speed
categories = ['rec.sport.baseball', 'sci.space']
data = fetch_20newsgroups(subset='train', categories=categories, shuffle=True, random_state=42)
test_data = fetch_20newsgroups(subset='test', categories=categories, shuffle=True, random_state=42)

pipe = make_pipeline(CountVectorizer(), MultinomialNB())
pipe.fit(data.data, data.target)
acc = accuracy_score(test_data.target, pipe.predict(test_data.data))
print(f"20 Newsgroups (baseball vs space) accuracy: {acc:.3f}")
print("\nNaive Bayes is a strong text classifier.")


## 16. Closed-Book Recall

Without looking back:

1. Write Bayes' theorem.
2. What is the naive assumption?
3. Why use log probabilities?
4. When does Naive Bayes fail?

## 17. Teach-Back Questions

Explain to another person:

- How Naive Bayes computes class probabilities.
- Why it works well for text.

## 18. Summary

You implemented Gaussian Naive Bayes from scratch, compared to sklearn, and used it for text classification. It's a fast, effective probabilistic classifier.

## 19. Further Experiment

- Try Bernoulli NB for binary features.
- Compare to logistic regression on text.

## 20. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, matplotlib, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
